# sportsdataverse-py quickstart

A cross-sport tour of the package — its layout, naming, and return types. Start here, then jump to the notebook for your sport.

**The shape of the package.** Every wrapper returns a raw `Dict` by default; opt into a tidy polars (or pandas) DataFrame with `return_parsed=True` (ESPN wrappers), a matching `parse_*` function (native NHL/MLB APIs), or by importing from the `sportsdataverse.parsed.<league>` mirror. Names follow a predictable pattern: `espn_<league>_<entity>()` for ESPN, `<league>_*` for a league's native API (NHL/MLB), and `load_<league>_*()` for pre-built parquet releases.

Part of the **[SportsDataverse](https://www.sportsdataverse.org)** — the names here mirror the R sisters (hoopR, wehoop, cfbfastR, baseballr, fastRhockey). See [Ecosystem & philosophy](https://py.sportsdataverse.org/docs/ecosystem) for the full picture.

## Setup

```sh
pip install sportsdataverse
# or
uv add sportsdataverse
```

In [ ]:
import polars as pl
import sportsdataverse as sdv

## Package layout

Each sport lives in its own submodule:

| Submodule | Coverage |
|-----------|----------|
| `sdv.cfb` | NCAA football: PBP, schedule, teams, play participants |
| `sdv.nfl` | NFL: nflverse parquet (PBP, schedules, NextGen, PFR) |
| `sdv.mbb` | NCAA men's basketball: PBP, schedule, rosters |
| `sdv.wbb` | NCAA women's basketball: PBP, schedule, rosters, stats, standings |
| `sdv.nba` | NBA: PBP, schedule, teams, rosters |
| `sdv.wnba` | WNBA: PBP, schedule, rosters, stats, standings, draft |
| `sdv.nhl` | NHL: PBP, schedule, teams |

In [ ]:
[m for m in dir(sdv) if not m.startswith('_')]

## Polars vs pandas

Every loader returns a polars `DataFrame` by default. Pass `return_as_pandas=True` to get a pandas frame instead — useful when downstream code (sklearn, statsmodels) expects pandas.

In [ ]:
teams_pl = sdv.wnba.espn_wnba_teams()
type(teams_pl).__name__, teams_pl.shape

In [ ]:
teams_pd = sdv.wnba.espn_wnba_teams(return_as_pandas=True)
type(teams_pd).__name__, teams_pd.shape

## The `download()` retry layer

All HTTP traffic goes through `sportsdataverse.dl_utils.download()` — a thin wrapper around `requests` with an exponential-style retry loop and ESPN-404 awareness. You can call it directly when you need a one-off endpoint that doesn't have a wrapper yet.

In [ ]:
from sportsdataverse.dl_utils import download

url = 'https://site.api.espn.com/apis/site/v2/sports/basketball/wnba/teams'
resp = download(url, num_retries=3)
resp.status_code, len(resp.content)

## Pipeline example: schedule -> first 5 games

Fetch a schedule, then pull the first PBP frame to confirm the round-trip works end-to-end.

In [ ]:
schedule = sdv.wnba.espn_wnba_schedule(dates=20240601)
schedule.select(['id', 'home_display_name', 'away_display_name', 'status_type_completed']).head()

## Cross-references

- R companion (umbrella): <https://www.sportsdataverse.org>
- Polars docs: <https://docs.pola.rs>
- pandas docs: <https://pandas.pydata.org/docs/>

## Where to go next

- `02_cfb_intro.ipynb` — college football
- `03_nfl_intro.ipynb` — NFL (nflverse parity surface)
- `04_nba_intro.ipynb` — NBA
- `05_wbb_wnba_intro.ipynb` — women's basketball (NCAA + WNBA)
- `06_mbb_intro.ipynb` — NCAA men's basketball
- `07_nhl_intro.ipynb` — NHL

Each notebook links back to its rendered API page under `docs/docs/<sport>/index.md`.